# CyberSOC GRPO Training
**Trains a SOC-analyst LLM using Group Relative Policy Optimization against the live CyberSOCEnv.**

**Recommended GPU**: A10G (24 GB) or A100 (40 GB)  
**Estimated runtime**: ~2–4 hours for 200 steps on A10G  
**Before running**: Set `HF_TOKEN` and `HUB_MODEL_ID` in Cell 3.

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    return result.returncode

# ── 1. Core HF / quantization stack ──────────────────────────────────────────
run('pip install -q --upgrade pip')
run('pip install -q "transformers>=4.45.0" "accelerate>=0.34.0" "peft>=0.13.0" "datasets>=2.21.0" "huggingface_hub>=0.25.0"')
run('pip install -q "bitsandbytes>=0.43.0"')

# ── 2. TRL — GRPO trainer (0.12 API is stable; cap at 0.14 to avoid breaking changes) ──
run('pip install -q "trl>=0.12.0,<0.15.0"')

# ── 3. Unsloth — efficient LoRA/4-bit (installs the right CUDA variant automatically) ──
run('pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"')

# ── 4. CyberSOC environment server dependencies ───────────────────────────────
run('pip install -q "fastapi>=0.115.0" "uvicorn[standard]>=0.30.0" "pydantic>=2.9.0" "requests>=2.31.0" "openenv-core" "aiofiles" "networkx" "websockets"')

# ── 5. Tensorboard for logging ────────────────────────────────────────────────
run('pip install -q tensorboard')

print('✅ All packages installed.')

## Cell 2 — Clone / locate the CyberSOC repo

In [ ]:
import os, sys

REPO_URL   = 'https://github.com/Ajayyy00/CyberSOC-upgraded.git'
REPO_DIR   = '/content/CyberSOC'          # change if not on Colab/HF

# If the repo is already present (e.g. running inside the HF Space), skip cloning.
if os.path.isdir(os.path.join(REPO_DIR, 'server')):
    print(f'Repo already present at {REPO_DIR}')
elif os.path.isdir(os.path.join(os.getcwd(), 'server')):
    REPO_DIR = os.getcwd()
    print(f'Using cwd as repo root: {REPO_DIR}')
else:
    import subprocess
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    print(f'Cloned to {REPO_DIR}')

# Add repo root to the Python path so all local imports work
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
print('Repo files:', os.listdir('.'))

## Cell 3 — Training Configuration  
*(edit these values before running)*

In [ ]:
import torch

# ── User settings ─────────────────────────────────────────────────────────────
HF_TOKEN      = ''          # your HuggingFace write token (or set via env var HF_TOKEN)
HUB_MODEL_ID  = ''          # e.g. 'yourname/cybersoc-soc-analyst-7b' (leave '' to skip push)
OUTPUT_DIR    = './checkpoints/cybersoc-grpo'

# ── Model ─────────────────────────────────────────────────────────────────────
# T4 (15 GB):   use 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
# A10G (24 GB): use 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'  ← default
# A100 (40 GB): use 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit'
gpu_mem_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0
print(f'GPU memory: {gpu_mem_gb} GB')

if gpu_mem_gb >= 38:
    MODEL_NAME = 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit'
elif gpu_mem_gb >= 20:
    MODEL_NAME = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
else:
    MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
print(f'Auto-selected model: {MODEL_NAME}')

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_R            = 16
LORA_ALPHA        = 16
LORA_DROPOUT      = 0.0
MAX_SEQ_LENGTH    = 4096
TARGET_MODULES    = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                     'gate_proj', 'up_proj', 'down_proj']

# ── GRPO ──────────────────────────────────────────────────────────────────────
MAX_STEPS                   = 200       # increase to 500+ for serious training
NUM_GENERATIONS             = 4         # completions sampled per prompt (reduce to 2 on T4)
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE               = 5e-6
WARMUP_RATIO                = 0.1
MAX_PROMPT_LENGTH           = 1024
MAX_COMPLETION_LENGTH       = 1536
TEMPERATURE                 = 0.9
TOP_P                       = 0.95
BETA                        = 0.001     # KL penalty
LOGGING_STEPS               = 5
SAVE_STEPS                  = 50
SEED                        = 42

# ── Environment ───────────────────────────────────────────────────────────────
ENV_PORT          = 8001               # avoid conflict with HF Space at 7860/8000
ENV_URL           = f'http://127.0.0.1:{ENV_PORT}'
TASK_IDS          = ['easy', 'medium', 'hard']
PROMPTS_PER_TASK  = 10                 # unique reset-prompts per difficulty tier

# Resolve token from environment if not set above
import os
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

print('Configuration ready.')

## Cell 4 — Start CyberSOC Environment Server

In [ ]:
import subprocess, time, sys, os, requests

def start_env_server(port: int, repo_dir: str) -> subprocess.Popen:
    env = os.environ.copy()
    env['CYBERSOC_ADAPTIVE'] = '1'
    env['PYTHONPATH']        = repo_dir  # ensure imports resolve

    proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'server.app:app',
         '--host', '127.0.0.1', '--port', str(port), '--log-level', 'warning'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=env,
        cwd=repo_dir,
    )
    return proc


def wait_for_server(url: str, timeout: int = 90) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            r = requests.get(f'{url}/health', timeout=3)
            if r.status_code == 200:
                return True
        except Exception:
            pass
        # Also accept a successful /reset response
        try:
            r = requests.post(f'{url}/reset', json={'task_id': 'easy'}, timeout=5)
            if r.status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(2)
    return False


# Kill any stale server on this port before starting a new one
import socket
def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

if _port_in_use(ENV_PORT):
    print(f'Port {ENV_PORT} already in use — assuming server is already running.')
    env_proc = None
else:
    print(f'Starting CyberSOC env server on port {ENV_PORT} ...')
    env_proc = start_env_server(ENV_PORT, REPO_DIR)
    ok = wait_for_server(ENV_URL, timeout=120)
    if not ok:
        err = env_proc.stderr.read(4096).decode(errors='replace')
        raise RuntimeError(f'Env server failed to start.\n{err}')
    print(f'✅ Env server healthy at {ENV_URL}')

## Cell 5 — Verify Server + Quick Smoke Test

In [ ]:
import requests, json

def smoke_test(env_url: str) -> None:
    for task in ['easy', 'medium', 'hard']:
        r = requests.post(f'{env_url}/reset', json={'task_id': task}, timeout=15)
        r.raise_for_status()
        obs = r.json().get('observation', r.json())
        alerts = len(obs.get('alert_queue', []))
        step   = obs.get('step_count', '?')
        print(f'  [{task:6s}] alerts={alerts}  step={step}  ✓')

print('Smoke-testing all three task difficulties ...')
smoke_test(ENV_URL)
print('\n✅ Server is healthy and returning valid observations.')

## Cell 6 — Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

print(f'Loading {MODEL_NAME} ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    load_in_4bit  = True,
    dtype         = None,    # auto-detect (bfloat16 on Ampere+)
    token         = HF_TOKEN or None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                        = LORA_R,
    target_modules           = TARGET_MODULES,
    lora_alpha               = LORA_ALPHA,
    lora_dropout             = LORA_DROPOUT,
    bias                     = 'none',
    use_gradient_checkpointing= 'unsloth',  # saves ~30% VRAM
    random_state             = SEED,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params    : {total_params/1e9:.2f}B')
print(f'Trainable params: {trainable_params/1e6:.1f}M  ({100*trainable_params/total_params:.2f}%)')
print('✅ Model loaded with LoRA adapters.')

## Cell 7 — Build Training Dataset

In [ ]:
import textwrap, requests
from datasets import Dataset

# ── System prompt (mirrors training/train_grpo.py) ────────────────────────────
SYSTEM_PROMPT = textwrap.dedent("""
    You are an expert SOC (Security Operations Center) analyst responding to a live
    cybersecurity incident. Given the current environment state, output a complete JSON
    array of investigation and containment actions, ending with submit_containment_plan.

    AVAILABLE ACTIONS — use exact field names:
      {\"type\": \"correlate_alerts\",          \"alert_ids\": [\"ID1\", \"ID2\"]}
      {\"type\": \"query_host\",                \"hostname\": \"HOSTNAME\"}
      {\"type\": \"run_forensics\",             \"hostname\": \"HOSTNAME\"}
      {\"type\": \"enrich_ioc\",               \"ioc_value\": \"VALUE\", \"ioc_type\": \"ip|domain|hash\"}
      {\"type\": \"scan_host_vulnerabilities\", \"hostname\": \"HOSTNAME\"}
      {\"type\": \"block_ioc\",                \"ioc_value\": \"VALUE\", \"ioc_type\": \"ip|domain|hash\"}
      {\"type\": \"kill_process\",             \"hostname\": \"HOSTNAME\", \"process_name\": \"PROCESS\"}
      {\"type\": \"isolate_segment\",          \"subnet\": \"SUBNET\", \"reason\": \"REASON\"}
      {\"type\": \"trigger_playbook\",         \"playbook_name\": \"NAME\", \"target\": \"HOSTNAME\"}
      {\"type\": \"submit_containment_plan\",  \"plan\": [
          {\"threat_id\": \"T-ID\", \"actions_taken\": [\"action1\"], \"root_cause\": \"...\", \"confidence\": 0.9}
        ], \"executive_summary\": \"TEXT\"}

    PLAYBOOK NAMES: ransomware_containment | c2_disruption | lateral_movement_lockdown |
                    phishing_response | data_exfil_stop

    GRADED ON 10 dimensions (evidence-gated — actions only score if prior evidence justifies them):
      1. TRIAGE   : correlate_alerts on related alerts
      2. INVESTIGATE: query_host then run_forensics on every alert source host
      3. ENRICH   : enrich_ioc on IOCs from alerts/forensics
      4. SCAN     : scan_host_vulnerabilities on confirmed-compromised hosts
      5. CONTAIN  : kill malicious processes; block relevant IOCs
      6. ISOLATE  : isolate_segment only for subnets with confirmed-compromised hosts
      7. REPORT   : submit_containment_plan as the FINAL action

    Output ONLY a valid JSON array. No markdown fences, no explanation.
""").strip()


def format_observation(obs: dict, task_id: str) -> str:
    """Convert a reset observation into a structured user-turn prompt."""
    alerts    = obs.get('alert_queue', [])
    topo      = obs.get('network_topology', {})
    threats   = obs.get('active_threats', [])
    playbooks = obs.get('available_playbooks', [])
    subnets   = ' | '.join(f'{k}:{v}' for k, v in topo.get('subnets', {}).items())

    alert_lines = []
    for a in alerts:
        iocs = ', '.join(a.get('ioc_indicators', []))
        line = (f"  [{a.get('severity','?').upper()}] {a.get('alert_id','?')} | "
                f"{a.get('source_host','?')} | {a.get('threat_type','?')} | "
                f"{a.get('description','')[:90]}")
        if iocs:
            line += f'\n    IOCs: {iocs}'
        alert_lines.append(line)

    return '\n'.join([
        f'TASK DIFFICULTY : {task_id.upper()}',
        f'MAX STEPS       : {obs.get("max_steps", 30)}',
        '',
        f'ALERT QUEUE ({len(alerts)} alerts):',
        *alert_lines,
        '',
        'NETWORK TOPOLOGY:',
        f'  Hosts: {topo.get("total_hosts","?"):} total | '
        f'{topo.get("compromised_count",0)} compromised | '
        f'{topo.get("isolated_count",0)} isolated',
        f'  Subnets: {subnets}',
        '',
        f'ACTIVE THREAT IDs : {", ".join(threats) if threats else "unknown — check alerts"}',
        f'AVAILABLE PLAYBOOKS: {", ".join(playbooks)}',
        '',
        'Generate your complete action sequence as a JSON array:',
    ])


def build_prompt(obs: dict, task_id: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': format_observation(obs, task_id)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


# ── Build dataset rows ────────────────────────────────────────────────────────
rows = []
for task_id in TASK_IDS:
    for i in range(PROMPTS_PER_TASK):
        try:
            r = requests.post(f'{ENV_URL}/reset', json={'task_id': task_id}, timeout=15)
            r.raise_for_status()
            obs = r.json().get('observation', r.json())
            rows.append({
                'prompt':  build_prompt(obs, task_id),
                'task_id': task_id,
            })
            print(f'  [{task_id}] prompt {i+1}/{PROMPTS_PER_TASK} ✓')
        except Exception as e:
            print(f'  [warn] {task_id} prompt {i+1} failed: {e}')

if not rows:
    raise RuntimeError('Dataset is empty — env server did not return any valid observations.')

dataset = Dataset.from_list(rows)
print(f'\n✅ Dataset: {len(dataset)} prompts across {TASK_IDS}')
print(f'   Columns : {dataset.column_names}')
print(f'\nSample prompt (truncated):\n{dataset[0]["prompt"][:600]}...')

## Cell 8 — Define Reward Functions

In [ ]:
import json, re, requests
from typing import List

DIMENSION_NAMES = [
    'threat_containment',
    'ioc_blocking',
    'forensic_investigation',
    'siem_correlation',
    'threat_intel_usage',
    'vuln_root_cause',
    'business_impact',
    'step_efficiency',
    'plan_coverage',
    'plan_evidence_quality',
]

WEIGHTS = {
    'threat_containment':     0.20,
    'ioc_blocking':           0.12,
    'forensic_investigation': 0.10,
    'siem_correlation':       0.08,
    'threat_intel_usage':     0.08,
    'vuln_root_cause':        0.08,
    'business_impact':        0.10,
    'step_efficiency':        0.07,
    'plan_coverage':          0.10,
    'plan_evidence_quality':  0.07,
}


def _extract_json(text: str) -> str:
    """Strip markdown fences or prose; extract the first JSON array."""
    text = text.strip()
    # Remove ```json ... ``` fences
    m = re.search(r'```(?:json)?\s*([\s\S]+?)```', text)
    if m:
        return m.group(1).strip()
    # Find first JSON array in raw text
    m = re.search(r'(\[[\s\S]+\])', text)
    if m:
        return m.group(1).strip()
    return text


def _run_episode(completion: str, task_id: str = 'hard',
                 env_url: str = ENV_URL, timeout: int = 45) -> dict | None:
    """Parse completion as JSON action list, replay against the env server.
    Returns the final observation dict, or None on any failure."""
    try:
        actions = json.loads(_extract_json(completion))
    except (json.JSONDecodeError, ValueError):
        return None
    if not isinstance(actions, list) or len(actions) == 0:
        return None

    try:
        r = requests.post(f'{env_url}/reset', json={'task_id': task_id}, timeout=timeout)
        if not r.ok:
            return None
        obs = r.json().get('observation', r.json())

        for action in actions:
            if not isinstance(action, dict) or 'type' not in action:
                continue
            r = requests.post(f'{env_url}/step', json=action, timeout=timeout)
            if not r.ok:
                break
            obs = r.json().get('observation', r.json())
            if obs.get('done', False):
                break

        return obs
    except Exception:
        return None


def _make_dim_fn(dimension: str):
    """Return a TRL-compatible reward function for one grading dimension."""
    def reward_fn(completions: List[str], **kwargs) -> List[float]:
        # TRL passes dataset columns as kwargs — pick task_id per sample
        raw_task_ids = kwargs.get('task_id', ['hard'] * len(completions))
        if isinstance(raw_task_ids, str):
            raw_task_ids = [raw_task_ids] * len(completions)
        scores = []
        for completion, tid in zip(completions, raw_task_ids):
            obs = _run_episode(completion, task_id=tid)
            if obs is None:
                scores.append(0.0)
                continue
            bd = obs.get('grade_breakdown') or obs.get('reward_dimensions') or {}
            scores.append(float(bd.get(dimension, 0.0)))
        return scores
    reward_fn.__name__ = f'soc_{dimension}'
    return reward_fn


def _make_weighted_total_fn():
    """Weighted composite across all 10 dimensions — the primary training signal."""
    def reward_fn(completions: List[str], **kwargs) -> List[float]:
        raw_task_ids = kwargs.get('task_id', ['hard'] * len(completions))
        if isinstance(raw_task_ids, str):
            raw_task_ids = [raw_task_ids] * len(completions)
        scores = []
        for completion, tid in zip(completions, raw_task_ids):
            obs = _run_episode(completion, task_id=tid)
            if obs is None:
                scores.append(0.0)
                continue
            bd = obs.get('grade_breakdown') or obs.get('reward_dimensions') or {}
            weighted = sum(bd.get(k, 0.0) * w for k, w in WEIGHTS.items())
            scores.append(float(weighted))
        return scores
    reward_fn.__name__ = 'soc_weighted_total'
    return reward_fn


# 10 per-dimension functions + 1 weighted composite
reward_fns = [_make_dim_fn(dim) for dim in DIMENSION_NAMES]
reward_fns.append(_make_weighted_total_fn())

print(f'✅ {len(reward_fns)} reward functions registered:')
for fn in reward_fns:
    print(f'   {fn.__name__}')

# ── Quick sanity-check: run one episode with a handcrafted perfect response ───
print('\nRunning reward sanity-check ...')
_test_obs_r = requests.post(f'{ENV_URL}/reset', json={'task_id': 'easy'}, timeout=15)
_test_obs   = _test_obs_r.json().get('observation', _test_obs_r.json())
_test_alerts= _test_obs.get('alert_queue', [])
_test_action = json.dumps([{
    'type': 'correlate_alerts',
    'alert_ids': [a['alert_id'] for a in _test_alerts[:2]] or ['A-001', 'A-002'],
}, {
    'type': 'run_forensics',
    'hostname': _test_alerts[0]['source_host'] if _test_alerts else 'WS-042',
}, {
    'type': 'submit_containment_plan',
    'plan': [{'threat_id': 'T-EASY-001', 'actions_taken': ['run_forensics'],
              'root_cause': 'test', 'confidence': 0.8}],
    'executive_summary': 'Test episode.',
}])
_scores = reward_fns[-1]([_test_action], task_id=['easy'])
print(f'   weighted_total for test episode = {_scores[0]:.4f}  (> 0 means reward functions work)')
assert _scores[0] >= 0, 'Reward functions returned negative score — check env server'
print('✅ Reward functions verified.')

## Cell 9 — Configure and Run GRPO Training

In [ ]:
from trl import GRPOTrainer, GRPOConfig
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Build GRPOConfig — handle minor API differences between TRL 0.12 and 0.14 ──
grpo_kwargs = dict(
    output_dir                   = OUTPUT_DIR,
    max_steps                    = MAX_STEPS,
    num_train_epochs             = 1,
    per_device_train_batch_size  = PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps  = GRADIENT_ACCUMULATION_STEPS,
    learning_rate                = LEARNING_RATE,
    warmup_ratio                 = WARMUP_RATIO,
    num_generations              = NUM_GENERATIONS,
    max_prompt_length            = MAX_PROMPT_LENGTH,
    max_completion_length        = MAX_COMPLETION_LENGTH,
    temperature                  = TEMPERATURE,
    top_p                        = TOP_P,
    beta                         = BETA,
    logging_steps                = LOGGING_STEPS,
    save_steps                   = SAVE_STEPS,
    seed                         = SEED,
    bf16                         = True,
    report_to                    = 'tensorboard',
    remove_unused_columns        = False,  # keep task_id column for reward fns
    dataloader_num_workers       = 0,      # avoids multiprocessing issues in notebooks
)

# Optional Hub push
if HUB_MODEL_ID and HF_TOKEN:
    grpo_kwargs.update(dict(
        push_to_hub      = True,
        hub_model_id     = HUB_MODEL_ID,
        hub_strategy     = 'checkpoint',
        hub_token        = HF_TOKEN,
    ))
    print(f'Hub push enabled → {HUB_MODEL_ID}')
else:
    grpo_kwargs['report_to'] = 'none'  # no hub, use no reporting to keep it simple
    print('Hub push disabled (set HUB_MODEL_ID and HF_TOKEN to enable).')

grpo_config = GRPOConfig(**grpo_kwargs)

# ── Instantiate trainer — try new API (processing_class) then fall back ───────
try:
    trainer = GRPOTrainer(
        model            = model,
        args             = grpo_config,
        processing_class = tokenizer,
        reward_funcs     = reward_fns,
        train_dataset    = dataset,
    )
    print('Trainer initialised with new API (processing_class).')
except TypeError:
    trainer = GRPOTrainer(
        model         = model,
        tokenizer     = tokenizer,
        reward_funcs  = reward_fns,
        args          = grpo_config,
        train_dataset = dataset,
    )
    print('Trainer initialised with legacy API (tokenizer).')

print(f'\n✅ Starting GRPO training for {MAX_STEPS} steps ...')
print(f'   Batch size effective: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * NUM_GENERATIONS}')
print(f'   Output dir: {OUTPUT_DIR}\n')

train_result = trainer.train()

print('\n' + '='*60)
print('Training complete!')
print(f'  Steps ran  : {train_result.global_step}')
print(f'  Train loss : {train_result.training_loss:.6f}')
print('='*60)

## Cell 10 — Save Model & Push to Hub

In [ ]:
import os

# ── Save LoRA adapters (fast, small — recommended for checkpointing) ──────────
lora_dir = os.path.join(OUTPUT_DIR, 'final-lora')
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print(f'LoRA adapters saved → {lora_dir}')

# ── Optionally save merged 16-bit weights (needed for vLLM inference) ─────────
SAVE_MERGED = False   # set True if you want the full merged model (~14 GB for 7B)
if SAVE_MERGED:
    merged_dir = os.path.join(OUTPUT_DIR, 'final-merged')
    model.save_pretrained_merged(
        merged_dir, tokenizer,
        save_method='merged_16bit',
    )
    print(f'Merged 16-bit model saved → {merged_dir}')

# ── Push to HF Hub ─────────────────────────────────────────────────────────────
if HUB_MODEL_ID and HF_TOKEN:
    print(f'\nPushing LoRA adapters to huggingface.co/{HUB_MODEL_ID} ...')
    model.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN)
    print(f'✅ Pushed to https://huggingface.co/{HUB_MODEL_ID}')
else:
    print('Skipping Hub push (HUB_MODEL_ID or HF_TOKEN not set).')

print('\n✅ All done.')

## Cell 11 — Cleanup (stop env server)

In [ ]:
if 'env_proc' in globals() and env_proc is not None:
    env_proc.terminate()
    try:
        env_proc.wait(timeout=5)
    except Exception:
        env_proc.kill()
    print('Env server stopped.')
else:
    print('No managed server process to stop.')

# Free GPU memory
import torch, gc
del trainer
gc.collect()
torch.cuda.empty_cache()
print('GPU cache cleared.')